# Post-processing of recorded spectra

Load the on/off spectrum pairs recorded by `rtlsdr-recorder`, clean and
downsample them, average them, and export the result to a FITS spectrum.

The recorder saves frequency-switched pairs of one-second integrations: an
*on* spectrum centered on the HI line at 1420 MHz and an *off* spectrum
centered 4 MHz below, so that the two bands do not overlap and the off band
is free of Galactic HI emission.

With the default `clip_difference=False`, the reduction consists of the
following steps:

1. Load all matched on/off spectrum pairs from the data directory (the
   recording settings are read from the `settings.json` saved alongside the
   data, falling back to the defaults for data recorded before settings
   files existed).
2. Mask RFI spikes in each on and each off spectrum by iterative sigma
   clipping.
3. Mask the channels at the center of each band, which always contain a
   spurious DC spike from the receiver.
4. Spectrally downsample each spectrum by block-averaging groups of
   adjacent channels, ignoring masked values.
5. Average all the on spectra and all the off spectra channel by channel,
   again ignoring masked values (channels masked in every spectrum, such as
   the DC region, become NaN).
6. Subtract the averaged off spectrum from the averaged on spectrum: since
   both were measured through the same receiver channels, this removes the
   bandpass shape and leaves the HI signal.
7. Optionally export the result to a FITS spectrum.

With `clip_difference=True`, the difference spectrum is instead reduced
from the per-pair differences:

1. Load all matched on/off spectrum pairs as above.
2. For each pair, subtract the off spectrum from the on spectrum, removing
   the bandpass shape.
3. Mask RFI spikes in each difference spectrum by iterative sigma clipping;
   because the differences are flat rather than dominated by the bandpass,
   this is more sensitive to RFI present in only one of the two bands.
4. Mask the channels at the center of the band as above.
5. Spectrally downsample each difference spectrum.
6. Average all the difference spectra channel by channel.
7. Optionally export the result to a FITS spectrum.

In both cases the returned on and off averages are always reduced with the
first list of steps; the flag only changes how the difference spectrum is
computed.

The cells below run the default steps one at a time; the whole reduction is
also available as a single call,
`rtlsdr_recorder.analysis.reduce_spectra('raw')`.

In [ ]:
from rtlsdr_recorder import frequency_array
from rtlsdr_recorder.analysis import (
    load_spectrum_pairs,
    clean_spectrum,
    downsample_spectrum,
    average_spectra,
    plot_spectrum_pair,
    write_fits,
)

Load all matched on/off spectrum pairs:

In [ ]:
spectra_on, spectra_off = load_spectrum_pairs('raw')
len(spectra_on)

Plot the first pair and its difference:

In [ ]:
plot_spectrum_pair(spectra_on[0], spectra_off[0], lw=0.1);

The spectra have RFI spikes, and there is always a spike at the center of
the band, so mask both with `clean_spectrum` (sigma clipping plus a mask on
the central channels):

In [ ]:
spectra_on = [clean_spectrum(spec) for spec in spectra_on]
spectra_off = [clean_spectrum(spec) for spec in spectra_off]
plot_spectrum_pair(spectra_on[0], spectra_off[0], lw=0.1);

Spectrally downsample the cleaned spectra:

In [ ]:
DOWNSAMPLE = 10
freq = downsample_spectrum(frequency_array(), DOWNSAMPLE)
spectra_on = [downsample_spectrum(spec, DOWNSAMPLE) for spec in spectra_on]
spectra_off = [downsample_spectrum(spec, DOWNSAMPLE) for spec in spectra_off]
plot_spectrum_pair(spectra_on[0], spectra_off[0], frequencies=freq);

That was the first pair only, so now average all pairs:

In [ ]:
average_on = average_spectra(spectra_on)
average_off = average_spectra(spectra_off)
plot_spectrum_pair(average_on, average_off, frequencies=freq);

Export the averaged difference spectrum to a FITS file:

In [ ]:
write_fits(average_on - average_off, 'spectrum.fits', frequencies=freq, overwrite=True)